# 04. Фиче-инжиниринг → модельный датасет

База: `eda_left.parquet` (чистые колонки, таргеты, district, row_number).
Таргет: **`is_sold_next_month`** (горизонт 30 дней). Утечка таргета (`price_deals*`, `registration_date_raw`,
`pledge_type`, `mortgage`, `unit_status`, `is_sold*`) в фичи НЕ идёт.

Группы фичей: 1) лот · 2) цена · 3) сегмент/конкуренция · 4) динамика цены · 5) время/жизненный цикл · 6) макро-фон.
Результат → `model_dataset.parquet` (в нём сохраняются ОБА таргета — `is_sold_next_month` и `is_sold_next_quarter`,
выбор горизонта делается в `05_training` / `src/config.TARGET_COL`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", 60)

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.config import EDA_LEFT_PARQUET, MODEL_DATASET_PARQUET

df = pd.read_parquet(EDA_LEFT_PARQUET)          # отсортирован по unit_match_key x file_date
print(f"{len(df):,} строк × {df.shape[1]} колонок")

4,631,451 строк × 40 колонок


## 1. Цена

In [2]:
df["log_price"] = np.log1p(df["price"])                       # стабилизирует тяжёлый правый хвост
df["discount_abs"] = df["price_no_discount"] - df["price"]    # абсолютная скидка, ₽/м²

## 2. Сегмент и конкуренция
Локальный сегмент: `file_date × district × project_class × room_count`.

In [3]:
seg = ["file_date", "district", "project_class", "room_count"]
gp = df.groupby(seg, observed=True)["price"]
df["seg_median_price"] = gp.transform("median")              # уровень цен в сегменте
df["seg_count"]        = gp.transform("size")                # активных лотов в сегменте-месяце
df["rel_price"]        = df["price"] / df["seg_median_price"] # цена относительно похожих (ключевой признак)
df["price_rank_seg"]   = gp.rank(pct=True)                   # ценовой перцентиль внутри сегмента

## 3. Динамика цены (по истории лота)
Лаги NaN на первом срезе лота (~19%) — это нормально, CatBoost обрабатывает NaN.

In [4]:
gu = df.groupby("unit_match_key", observed=True)
price_prev = gu["price"].shift(1)
df["price_change_pct_1"]      = df["price"] / price_prev - 1            # к прошлому срезу
df["price_change_from_first"] = df["price"] / gu["price"].transform("first") - 1   # к первой цене
df["discount_change_1"]       = df["discount_pct"] - gu["discount_pct"].shift(1)
df["days_since_prev"]         = (df["file_date"] - gu["file_date"].shift(1)).dt.days
_changed = ((df["price"] != price_prev) & price_prev.notna()).astype("int8")
df["n_price_changes"]         = _changed.groupby(df["unit_match_key"]).cumsum()   # число изменений цены к этому срезу


## 4. Время и жизненный цикл

In [5]:
df["file_year"]  = df["file_date"].dt.year.astype("int16")
df["file_month"] = df["file_date"].dt.month.astype("int8")
df["days_since_sales_start"] = (df["file_date"] - df["sales_start"]).dt.days
df["months_to_completion"]   = ((df["completion_k"] - df["file_date"]).dt.days / 30.44).round(1)

# флаг лево-усечения: на первом срезе лот уже был экспонирован (часть истории вне окна)
first_exp = df.loc[df["row_number"] == 1].set_index("unit_match_key")["exposure"].astype("float64")
df["history_truncated"] = (df["unit_match_key"].map(first_exp) > 0).astype("int8")

## 5. Макроэкономический фон (матч по месяцу `file_date`)

Ключевая ставка ЦБ и инфляция (г/г) из `interest_rate_and_inflation.xlsx`, сопоставляются по месяцу прайс-среза.


In [6]:
from src.config import MACRO_XLSX

raw = pd.read_excel(MACRO_XLSX)                      # 'Дата' хранится как float M.YYYY
v = raw["Дата"].astype(float)
mo = np.floor(v).astype(int)
yr = ((v - mo) * 10000).round().astype(int)
macro = pd.DataFrame({
    "ym":        pd.to_datetime(dict(year=yr, month=mo, day=1)),
    "key_rate":  raw["Ключевая ставка, % годовых"].astype(float),
    "inflation": raw["Инфляция, % г/г"].astype(float),
}).set_index("ym")
macro["real_rate"] = macro["key_rate"] - macro["inflation"]   # реальная ставка, п.п.

# матч по месяцу file_date (через map -> порядок строк не меняется)
ym = df["file_date"].dt.to_period("M").dt.start_time
df["key_rate"]  = ym.map(macro["key_rate"])
df["inflation"] = ym.map(macro["inflation"])
df["real_rate"] = ym.map(macro["real_rate"])

print("NaN после матча:",
      {c: int(df[c].isna().sum()) for c in ["key_rate", "inflation", "real_rate"]})


NaN после матча: {'key_rate': 0, 'inflation': 0, 'real_rate': 0}


## 6. Геофичи окружения ЖК (OSM, по `project_id`)

Геопризнаки считаются один раз на проект в `04a_geo_features` (метро, школы,
зелень, вода, промзоны, кладбища, ЛЭП, дороги, ж/д, плотность соседних ЖК) и
сохраняются в `geo_features.parquet`. Здесь мерджим их в датасет по `project_id`.

In [20]:
import src.geo_features as geo
from src.config import GEO_FEATURES_PARQUET, OSM_CACHE_DIR

if GEO_FEATURES_PARQUET.exists():
    geo_df = pd.read_parquet(GEO_FEATURES_PARQUET)
else:
    print("geo_features.parquet нет — считаю на лету (запусти 04a, чтобы кэшировать)…")
    geo_df = geo.compute_geo_features(df, OSM_CACHE_DIR)
    GEO_FEATURES_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    geo_df.to_parquet(GEO_FEATURES_PARQUET)

GEO_FEATURES = geo.feature_columns()
df = df.merge(geo_df.reset_index(), on="project_id", how="left")

coverage = 100 * df["dist_to_metro_m"].notna().mean()
print(f"Геофич добавлено: {len(GEO_FEATURES)} | проектов с гео: {geo_df.shape[0]}")
print(f"Покрытие строк (есть координаты проекта): {coverage:.1f}%")


Геофич добавлено: 35 | проектов с гео: 752
Покрытие строк (есть координаты проекта): 99.9%


## 6b. Рыночные фичи (состояние сегмента и его динамика на момент среза)
Cross-section агрегаты — с `file_date` в ключе (point-in-time). Динамика и absorption — строго прошлые месяцы (`shift`+`rolling`, текущий исключён).

In [8]:
# A. разброс и позиция цены внутри сегмента-среза (point-in-time)
gp = df.groupby(seg, observed=True)["price"]                       # seg = file_date+district+class+room (cell 5)
_mean, _std = gp.transform("mean"), gp.transform("std")
df["seg_price_iqr"]    = gp.transform(lambda s: s.quantile(.75) - s.quantile(.25))
df["seg_price_cv"]     = _std / _mean                              # неоднородность сегмента
df["price_gap_to_p10"] = df["price"] / gp.transform(lambda s: s.quantile(.10)) - 1  # дороже дешёвых конкурентов

# B. предложение и конкуренция (число активных лотов на срезе)
df["n_active_project"]  = df.groupby(["project_id", "file_date"], observed=True)["price"].transform("size")
df["n_active_building"] = df.groupby(["building_id", "file_date"], observed=True)["price"].transform("size")
df["n_active_district"] = df.groupby(["district", "file_date"], observed=True)["price"].transform("size")
df["competitor_density"] = gp.transform("size") / df["n_active_district"]            # доля сегмента в районе
df["room_share_project"] = (df.groupby(["project_id", "room_count", "file_date"], observed=True)["price"].transform("size")
                            / df["n_active_project"])               # редкая/затоваренная планировка в ЖК

# F. возраст и застой предложения в сегменте-срезе
_age = (df["file_date"] - df["sales_start"]).dt.days
df["seg_median_exposure"] = df.groupby(seg, observed=True)["exposure"].transform("median")
df["rel_exposure"]        = df["exposure"] / df["seg_median_exposure"]               # вишу дольше соседей?
df["seg_median_age"]      = _age.groupby([df[c] for c in seg]).transform("median")
df["share_stale_seg"]     = (df["exposure"] > 180).groupby([df[c] for c in seg]).transform("mean")

# H. позиция относительно более широких рынков
_distr = df.groupby(["district", "file_date"], observed=True)["price"].transform("median")
df["rel_price_district"] = df["price"] / _distr
df["seg_premium"]        = df.groupby(seg, observed=True)["price"].transform("median") / _distr  # премия микро-сегмента к району
df["dev_price_premium"]  = df.groupby(["developer", "file_date"], observed=True)["price"].transform("median") / _distr

# доля свежих листингов в сегменте-срезе (приток предложения)
df["share_new_supply"] = (df["row_number"] == 1).groupby([df[c] for c in seg]).transform("mean")

In [9]:
# C/D. динамика сегмента через панель (SEG x месяц); текущий месяц исключаем (shift перед rolling)
df["ym"] = df["file_date"].dt.to_period("M")
seg_m = ["district", "project_class", "room_count"]

panel = (df.groupby(seg_m + ["ym"], observed=True)
           .agg(_n=("price", "size"), _med=("price", "median")).reset_index())

# поток продаж: лот регистрируется один раз -> агрегируем по месяцу регистрации (история закрытых)
sold = (df.loc[df["registration_date_raw"].notna(), ["unit_match_key"] + seg_m + ["registration_date_raw"]]
          .drop_duplicates("unit_match_key"))
sold["ym"] = sold["registration_date_raw"].dt.to_period("M")
sold_m = sold.groupby(seg_m + ["ym"], observed=True).size().rename("_sold").reset_index()
panel = panel.merge(sold_m, on=seg_m + ["ym"], how="left")
panel["_sold"] = panel["_sold"].fillna(0)

panel = panel.sort_values(seg_m + ["ym"])
g = panel.groupby(seg_m, observed=True, sort=False)
# C — лаги предложения и уровня цен (shift(k) = k срезов назад)
panel["seg_count_chg_1m"]        = panel["_n"]   / g["_n"].shift(1)   - 1
panel["seg_count_chg_3m"]        = panel["_n"]   / g["_n"].shift(3)   - 1
panel["seg_median_price_chg_1m"] = panel["_med"] / g["_med"].shift(1) - 1
panel["seg_median_price_chg_3m"] = panel["_med"] / g["_med"].shift(3) - 1   # ценовой моментум сегмента
panel["seg_median_price_chg_6m"] = panel["_med"] / g["_med"].shift(6) - 1
# D — absorption / месяцы запаса (trailing по прошлым месяцам, текущий исключён)
panel["seg_sold_prev_3m"]    = g["_sold"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())
_active_prev_3m              = g["_n"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
panel["seg_absorption_3m"]   = panel["seg_sold_prev_3m"] / _active_prev_3m   # скорость вымывания
_sold_prev_6m                = g["_sold"].transform(lambda s: s.shift(1).rolling(6, min_periods=1).sum())
panel["months_of_inventory"] = panel["_n"] / (_sold_prev_6m / 6)             # месяцы запаса

cd_cols = ["seg_count_chg_1m", "seg_count_chg_3m", "seg_median_price_chg_1m",
           "seg_median_price_chg_3m", "seg_median_price_chg_6m",
           "seg_sold_prev_3m", "seg_absorption_3m", "months_of_inventory"]
df = df.merge(panel[seg_m + ["ym"] + cd_cols], on=seg_m + ["ym"], how="left").drop(columns="ym")

In [10]:
# G. макро-моментум (лаги ключевой ставки и инфляции)
mac = (df[["file_date", "key_rate", "inflation", "real_rate"]]
       .drop_duplicates("file_date").sort_values("file_date"))
mac["key_rate_chg_3m"]  = mac["key_rate"]  - mac["key_rate"].shift(3)
mac["key_rate_chg_6m"]  = mac["key_rate"]  - mac["key_rate"].shift(6)
mac["real_rate_chg_3m"] = mac["real_rate"] - mac["real_rate"].shift(3)
mac["inflation_chg_3m"] = mac["inflation"] - mac["inflation"].shift(3)
df = df.merge(mac[["file_date", "key_rate_chg_3m", "key_rate_chg_6m", "real_rate_chg_3m", "inflation_chg_3m"]],
              on="file_date", how="left")

# деления на 0 -> inf -> NaN (CatBoost ест NaN, но не inf)
_mkt_ratio = ["seg_price_cv", "price_gap_to_p10", "competitor_density", "room_share_project",
              "rel_exposure", "rel_price_district", "seg_premium", "dev_price_premium",
              "seg_count_chg_1m", "seg_count_chg_3m", "seg_median_price_chg_1m", "seg_median_price_chg_3m",
              "seg_median_price_chg_6m", "seg_absorption_3m", "months_of_inventory"]
df[_mkt_ratio] = df[_mkt_ratio].replace([np.inf, -np.inf], np.nan)
print("Рыночных фич добавлено: 28")

Рыночных фич добавлено: 28


## 7. Сборка модельного датасета

In [16]:
GEO_FEATURES

[]

In [22]:
TARGET = "is_sold_next_month"

NUM_FEATURES = [
    # лот
    "area", "floor",
    # цена
    "price", "log_price", "price_no_discount", "discount_pct", "discount_abs",
    # сегмент / конкуренция
    "rel_price", "seg_median_price", "price_rank_seg", "seg_count",
    # динамика цены
    "price_change_pct_1", "price_change_from_first", "discount_change_1",
    "days_since_prev", "n_price_changes",
    # время / жизненный цикл
    "exposure", "row_number", "days_since_sales_start", "months_to_completion",
    "file_month", "file_year", "history_truncated",
    # макро-фон (матч по месяцу file_date)
    "key_rate", "inflation", "real_rate",
]
# геофичи окружения ЖК (OSM), смерджены по project_id в секции 6
NUM_FEATURES = NUM_FEATURES + GEO_FEATURES
# рыночные фичи (блоки A,B,C,D,F,G,H из секции 6b)
MARKET_FEATURES = [
    "seg_price_iqr", "seg_price_cv", "price_gap_to_p10",
    "n_active_project", "n_active_building", "n_active_district", "competitor_density", "room_share_project",
    "seg_count_chg_1m", "seg_count_chg_3m", "seg_median_price_chg_1m", "seg_median_price_chg_3m",
    "seg_median_price_chg_6m", "share_new_supply",
    "seg_sold_prev_3m", "seg_absorption_3m", "months_of_inventory",
    "seg_median_exposure", "rel_exposure", "seg_median_age", "share_stale_seg",
    "key_rate_chg_3m", "key_rate_chg_6m", "real_rate_chg_3m", "inflation_chg_3m",
    "rel_price_district", "seg_premium", "dev_price_premium",
]
NUM_FEATURES = NUM_FEATURES + MARKET_FEATURES

CAT_FEATURES = [
    "project_class", "region", "macro_district", "district",
    "room_count", "premises_type", "finish_tier", "finish_in_report",
    "unit_typology", "stage_k", "contract_type_k", "developer", "project_name",
]

# защита от утечки таргета
LEAK = {"price_deals", "price_deals_no_discount", "registration_date_raw",
        "pledge_type", "mortgage", "unit_status",
        "is_sold", "is_sold_past", "is_sold_next_month", "is_sold_next_quarter"}
assert not (set(NUM_FEATURES + CAT_FEATURES) & LEAK), "В фичи попала leak-колонка!"

# категориальные -> string (для CatBoost)
for c in CAT_FEATURES:
    df[c] = df[c].astype("string")

KEYS = ["unit_match_key", "file_date"]
KEEP_TARGETS = ["is_sold", "is_sold_next_month", "is_sold_next_quarter", "is_sold_past", "unit_status"]
model = df[KEYS + NUM_FEATURES + CAT_FEATURES + KEEP_TARGETS].copy()

MODEL_DATASET_PARQUET.parent.mkdir(parents=True, exist_ok=True)
model.to_parquet(MODEL_DATASET_PARQUET, index=False)

print(f"Сохранено: {MODEL_DATASET_PARQUET.name}  |  {model.shape[0]:,} × {model.shape[1]}")
print(f"Признаков: {len(NUM_FEATURES)} числовых + {len(CAT_FEATURES)} категориальных")
print(f"Таргет {TARGET}: {100*model[TARGET].mean():.2f}% положительных")
print("\nNaN% по числовым (NaN ок для CatBoost):")
display((100 * model[NUM_FEATURES].isna().mean()).round(1).sort_values(ascending=False).to_frame("NaN%"))

Сохранено: model_dataset.parquet  |  4,631,451 × 109
Признаков: 89 числовых + 13 категориальных
Таргет is_sold_next_month: 1.67% положительных

NaN% по числовым (NaN ок для CatBoost):


,NaN%
price_change_pct_1,19.1
days_since_prev,19.1
discount_change_1,19.1
rel_exposure,16.6
months_of_inventory,12.0
...,...
inflation,0.0
real_rate,0.0
floor,0.0
seg_price_iqr,0.0
